In [ ]:
# Colab setup - run this first, then switch the runtime to R
# (Runtime > Change runtime type > R). Precompiled Linux binaries via Posit P3M
# keep the install to a couple of minutes instead of compiling from source.
local({
  cn <- tryCatch(system("lsb_release -cs", intern = TRUE), error = function(e) "jammy")
  options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", cn)),
          HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
            paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
})
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(
  c("SingleCellExperiment", "SummarizedExperiment", "MultiAssayExperiment",
    "S4Vectors", "basilisk"),
  update = FALSE, ask = FALSE)
install.packages(c("remotes", "reticulate", "Matrix", "ggplot2"))
remotes::install_github("PYangLab/M3", subdir = "m3-r")
# The first m3_train() builds the bundled Python engine via basilisk (a few
# minutes on first run); afterwards it is cached for the session.

# Cell-level representation learning

The goal: turn raw, multi-batch, multi-modal single-cell data into one clean **embedding**, a handful of coordinates per cell that mix the batches together while keeping the biology, so you can cluster and analyze all your data as one.

This runs on the built-in **Liu et al.** COVID-19 subsample (~30k cells, 3 batches, RNA + ADT).

In [ ]:
library(m3)
library(ggplot2)
set.seed(0)

## 1. Load the demo dataset

`m3_demo()` returns a ready-to-use `m3_dataset`, a subsample of the Liu et al. COVID-19 CITE-seq data (RNA + ADT), with the obs columns already set up.

In [ ]:
data <- m3_demo()
data

## 2. Build and train the model

`m3_train(...)` sets up the model from your data and fits it, telling it which obs columns to use:

- `condition_keys`, the conditions you're comparing across (here disease group + age band).
- `celltype_key`, the column holding cell-type labels.
- `batch_key`, the column marking which batch each cell is from (B1/B2/B3). The model corrects for batch, so cells group by biology rather than by batch.
- `embedding_dim`, how many numbers describe each cell in the output (here 30).

`donor_prediction = FALSE` fits only the integration model and skips the disease-prediction step (not needed for embeddings, and faster). `seed = 0` makes the run reproducible. Afterwards `m3_capabilities(model)` shows which outputs are available.

In [ ]:
#To save time, users can set max_epochs to 100 for test.
model <- m3_train(
  data,
  condition_keys = c("cond_group", "Age_interval"),
  celltype_key   = "mergedcelltype",
  batch_key      = "batch",
  embedding_dim  = 30L,
  donor_prediction = FALSE,
  max_epochs = 500L,
  seed = 0L
)

model

m3_capabilities(model)

## 3. Get the embedding

`m3_embedding(model, part = ...)` returns the trained coordinates as a cells x embedding_dim matrix. Ask for the part you want by name:

- `"bio"`, the biological signal, with batch differences removed. This is the one you cluster and analyze.
- `"batch"`, the batch signal on its own, if you ever want to inspect it separately.

`m3_cell_metadata(model)` is the per-cell obs table, row-aligned with the embedding.

In [ ]:
emb_bio       <- m3_embedding(model, part = "bio")
emb_intrinsic <- m3_embedding(model, part = "intrinsic")
emb_batch     <- m3_embedding(model, part = "batch")
meta <- m3_cell_metadata(model)
cat(sprintf("bio: %s | intrinsic: %s | batch: %s\n",
            paste(dim(emb_bio), collapse = "x"),
            paste(dim(emb_intrinsic), collapse = "x"),
            paste(dim(emb_batch), collapse = "x")))

## 4. Save the embedding + metadata

Save the embedding and its metadata. m3 returns plain matrices, so persist them however you like, here as CSV and an `.rds`.

In [ ]:
out <- file.path(tempdir(), "m3_tut1")
dir.create(out, showWarnings = FALSE)
utils::write.csv(emb_bio, file.path(out, "embedding_bio.csv"), row.names = FALSE)
utils::write.csv(meta, file.path(out, "cell_metadata.csv"), row.names = FALSE)
saveRDS(list(embedding = emb_bio, metadata = meta), file.path(out, "embedding_bio.rds"))
cat("saved embedding to", out, "\n")

## 5. Visualise with UMAP

A UMAP of the `"bio"` embedding. If integration worked, cells should group by **cell type** (biology kept) while the **batches mix together** (batch difference removed), colour by each to check.

In [ ]:
xy <- m3_umap(emb_bio, method = "scanpy", n_neighbors = 15L, device = model$device)
df <- data.frame(UMAP1 = xy[, 1], UMAP2 = xy[, 2],
                 celltype = factor(meta$mergedcelltype),
                 batch    = factor(meta$batch),
                 condition = factor(meta$cond_group))

plt <- function(colour, title, legend = TRUE) {
  ggplot(df, aes(UMAP1, UMAP2, colour = .data[[colour]])) +
    geom_point(size = 0.4, alpha = 0.8) +
    theme_classic() +
    theme(axis.text = element_blank(), axis.ticks = element_blank(),
          legend.position = if (legend) "right" else "none") +
    guides(colour = guide_legend(override.aes = list(size = 2))) +
    labs(title = title, colour = NULL)
}
print(plt("celltype",  "Cell type (biology preserved)"))

In [ ]:
print(plt("batch",     "Batch (batch mixed)"))

In [ ]:
print(plt("condition", "Condition"))

**Done.** You now have an integrated cell embedding, ready for clustering and downstream analysis. The patient-prediction tutorial builds on the same model to predict donor-level disease status.